In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx

#==============================================================================
# 1. 全局参数 & H₂ 分子定义
#==============================================================================
K = 2  # NES-VMC 要计算的低激发态数量（基态+1个激发态）
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准（4个态）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# 转为 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

#==============================================================================
# 2. Hilbert 空间定义
#==============================================================================
n_orbitals = 2
n_spin_orbitals = 4
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

#==============================================================================
# 3. NES-VMC 神经网络模型（复数 FFNN）
#==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        #log_out = jnp.log(out)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        log_psi_total = jnp.log(psi_total)
        return psi_total, M

# 初始化模型
rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(n_spin_orbitals, n_states=K, hidden_dim=16, rngs=rngs)

#==============================================================================
# 4. 核心：哈密顿作用 + 局部能量矩阵（论文标准）
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz, x: jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    K_state = total_ansatz.n_states
    H_mat = []
    for i in range(K_state):
        row = []
        for j in range(K_state):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch, return_log_psi=False):
    psi, M = total_ansatz(x_batch, return_log_psi=return_log_psi)
    # 🔥 修复 1：对角加载，防止矩阵奇异
    eps = 1e-3
    M = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M, Hp)
    return jnp.trace(el_mat), el_mat

#==============================================================================
# 5. 损失函数 & 训练步骤（JAX 自动求导）
#==============================================================================
@jax.jit
def loss_fn(model_state, samples):
    nnx.update(total_ansatz, model_state)
    total_energy = 0.0 + 0j
    n_samples = samples.shape[0]
    for xb in samples:
        tr_EL, _ = compute_local_energy(ha, total_ansatz, xb)
        total_energy += tr_EL
    avg_energy = total_energy.real / n_samples
    return avg_energy

@jax.jit
def train_step(model_state, samples):
    loss, grads = jax.value_and_grad(loss_fn)(model_state, samples)
    model_state = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, model_state, grads)
    return model_state, loss



/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [2]:
#==============================================================================
# 【单样本版本】完全使用你的 API，100% 你的逻辑
#==============================================================================
def compute_local_energy_single(ham: nk.operator.DiscreteOperator, 
                               model: NESTotalAnsatz, 
                               x_batch):
    # x_batch: (K, n_spin_orbitals)
    tr_el, el_mat = compute_local_energy(ham, model, x_batch)
    return tr_el.real, el_mat  # 确保返回实数迹


#==============================================================================
# 【修复版 VMAP】参数数量匹配！in_axes 长度 = 参数个数
#==============================================================================
compute_local_energy_batch = jax.vmap(
    compute_local_energy_single,
    # 对应 4 个入参：ham, model, x_batch, return_log_psi
    in_axes=(None, None, 0),
    out_axes=(0, 0),
    axis_name='samples_batch'
)
#==============================================================================
# 【自动平均版本】输入 samples = (K, batch, n_orb)
#==============================================================================
def compute_average_local_energy(ham: nk.operator.DiscreteOperator, 
                                 model: NESTotalAnsatz, 
                                 samples):
    tr_els, el_mats = compute_local_energy_batch(
        ham, model, samples
    )
    
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    
    return tr_avg, el_mat_avg


def forward(state, x_batch):
    # 正确解包：log_psi_total 是数值，new_state 是状态
    (psi_total,_), new_state = nnx.call(state)(x_batch)
    
    # 直接用 数值 log_psi_total 填充，完美兼容 jnp.full
    return jnp.full((K,), log_psi_total)

In [4]:
# 采样器 n_chains=K
g = nk.graph.Graph(edges=[(0,1), (2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=K, spin_symmetric=True, sweep_size=64
)

# 完全符合论文：采样 |TotalAnsatz|²
def forward(state, x_batch):
    # 正确解包：log_psi_total 是数值，new_state 是状态
    (psi_local,_), new_state = nnx.call(state)(x_batch)
    
    # 直接用 数值 log_psi_total 填充，完美兼容 jnp.full
    return jnp.full((K,), psi_local)

parameters = nnx.split(total_ansatz)

sampler_state = sampler.init_state(forward, parameters, seed=1)
sampler_state = sampler.reset(forward, parameters, sampler_state)

samples, sampler_state = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=500
)
#samples.shape = (K,batch_size,n_spin_orbitals) = (2,500,4)

In [5]:
x_batch =jnp.array([[1,0,1,0],[0,1,0,1]])
compute_local_energy(ha,total_ansatz, x_batch)

TypeError: NESTotalAnsatz.__call__() got an unexpected keyword argument 'return_log_psi'

In [ ]:
compute_local_energy_single(ha,total_ansatz, x_batch,False)
        

In [ ]:
compute_local_energy_batch = jax.vmap(
    compute_local_energy_single,
    # 对应 4 个入参：ham, model, x_batch, return_log_psi
    in_axes=(None, None, 1, None),
    out_axes=(0, 0),
    axis_name='samples_batch'
)
a,b = compute_local_energy_batch(ha, total_ansatz, samples,False)

In [ ]:
# samples.shape = (2, 500, 4)
tr_avg, el_mat_avg = compute_average_local_energy(
    ha, total_ansatz, samples, return_log_psi=False
)

In [ ]:
#==============================================================================
# 【包装函数】仅对 model 参数求导
#==============================================================================
def loss_fn(params, graph, ham, x_batch):
    # 把参数放回模型
    model = nnx.merge(graph, params)
    # 计算 loss = trace(local energy matrix)
    tr_el, _ = compute_local_energy_single(ham, model, x_batch)
    return tr_el  # 我们对这个标量求导！

In [ ]:
# 生成梯度函数
val_and_grad_fn = jax.value_and_grad(loss_fn)

In [ ]:
# 先把 model 拆成 graph + params
graph, params = nnx.split(total_ansatz)

# 计算 value + grad
tr_value, grads = val_and_grad_fn(
    params,         # 对这个求导
    graph,
    ha,             # ham 算子
    x_batch         # shape=(K, n_spin) = (2,4)
)

In [ ]:
def loss_fn(params, graph, ham, samples):
    model = nnx.merge(graph, params)
    tr_avg, _ = compute_average_local_energy(ham, model, samples)
    return tr_avg  # ✅ 这是【平均值】

value_and_grad = jax.value_and_grad(loss_fn)
loss, grads = value_and_grad(params, graph, ha, samples)

In [ ]:
import jax
import flax.nnx as nnx
import optax

#==============================================================================
# 【批量平均损失】对整批 samples 求平均迹
#==============================================================================
def loss_fn(params, graph, ham, samples):
    # 合并参数 -> 模型
    model = nnx.merge(graph, params)
    # 计算【整批平均】loss = Tr( <E_mat> )
    tr_avg, _ = compute_average_local_energy(ham, model, samples)
    return tr_avg

#==============================================================================
# 生成梯度函数：自动对【平均值】求导
#==============================================================================
value_and_grad_fn = jax.value_and_grad(loss_fn)

# 拆分模型：graph 结构 + params 可训练参数
graph, params = nnx.split(total_ansatz)

# 优化器（Adam 稳定）
optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

In [ ]:
def train_step(params, opt_state, graph, ham, samples):
    # 1. 计算【平均损失】 + 【平均梯度】
    loss, grads = value_and_grad_fn(params, graph, ham, samples)
    
    # 2. 优化器更新参数
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    return params, opt_state, loss

In [ ]:
# 超参
num_steps = 700

print("="*60)
print("开始 NES-VMC 训练（批量平均梯度版）")
print("="*60)

for step in range(num_steps):
    # 采样（你已有的采样结果）
    # samples.shape = (K, batch, n_spin) = (2,500,4)
    
    # 训练一步
    params, opt_state, loss = train_step(
        params, opt_state, graph, ha, samples
    )
    
    # 打印
    if step % 50 == 0:
        print(f"Step {step:4d} | Loss = {loss:.6f} Ha")

# 训练结束，把参数放回模型
total_ansatz = nnx.merge(graph, params)

In [ ]:
tr_avg, el_mat_avg = compute_average_local_energy(ha, total_ansatz, samples)
energies = jnp.linalg.eigvalsh(el_mat_avg).real

print("\n" + "="*60)
print("✅ 最终 NES-VMC 能量")
print("="*60)
print(f"E0 = {energies[0]:.8f} Ha (基态)")
print(f"E1 = {energies[1]:.8f} Ha (激发态)")
print(f"FCI 基态 = {E_fcis[0]:.8f} Ha")
print(f"FCI 基态 = {E_fcis[1]:.8f} Ha")

In [ ]:
#==============================================================================
# 用采样结果计算局部能量（你直接复制运行）
#==============================================================================
K = total_ansatz.n_states

# 初始化矩阵累加
el_mat_sum = jnp.zeros((K, K), dtype=jnp.complex128)
tr_sum = 0.0
n_samples = samples.shape[1]  # 500

# 遍历每一组样本
for b in range(n_samples):
    x_batch = samples[:, b, :]   # shape = (K, 4) → 这就是论文的 x¹,x²
    tr_el, el_mat = compute_local_energy(total_ansatz, x_batch)
    
    tr_sum += tr_el
    el_mat_sum += el_mat

# 平均值
tr_avg = tr_sum / n_samples
el_mat_avg = el_mat_sum / n_samples

# 对角化 → 得到基态 & 激发态能量
eigen_energies = jnp.linalg.eigvalsh(el_mat_avg).real

print("="*60)
print("局部能量迹（训练Loss）:", tr_avg)
print("平均局部能量矩阵:")
print(el_mat_avg)
print("="*60)
print("对角化得到的 NES-VMC 能量：")
for i, e in enumerate(eigen_energies):
    print(f"E{i} = {e:.8f} Ha")

In [ ]:
samples[:,0,:]

In [ ]:
compute_local_energy(ha=ha,total_ansatz=total_ansatz,x_batch=samples[:,0,:])

In [ ]:
compute_local_energy(ha=ha,total_ansatz=total_ansatz,x_batch=samples[:,0,:])

In [ ]:
parameters = nnx.split(total_ansatz)
for i in range(50):
    
    sampler_state = sampler.init_state(forward, parameters, seed=1)
    sampler_state = sampler.reset(forward, parameters, sampler_state)
    samples, sampler_state = sampler.sample(
        forward, parameters, state=sampler_state, chain_length=500
    )
    
    

In [ ]:
#==============================================================================
# 6. 训练循环 + 采样 + 能量对角化（最后一部分！）
#==============================================================================
print("\n" + "="*60)
print(f"开始 NES-VMC 训练（K={K} 个态）| 采样 = Total Ansatz Ψ²")
print("="*60)

# 训练参数
n_steps = 800
lr = 0.005
model_state = nnx.state(total_ansatz)

# 主训练循环
for step in range(n_steps):
    # 1. 采样：从 Total Ansatz 联合分布采样
    samples, sampler_state = sampler.sample(
        forward, parameters, state=sampler_state, chain_length=500
    )
    
    # 2. 训练一步
    model_state, loss = train_step(model_state, samples)
    
    # 3. 日志输出
    if step % 50 == 0:
        print(f"Step {step:4d} | Loss (Tr(EL)) = {loss:.8f} Ha")

#==============================================================================
# 7. 最终计算：平均局部能量矩阵 + 对角化得到激发态
#==============================================================================
print("\n" + "="*60)
print("最终结果：对角化局部能量矩阵 → 激发态能量")
print("="*60)

# 加载最新模型参数
nnx.update(total_ansatz, model_state)

# 批量采样用于平均能量矩阵
samples_final, _ = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=500
)

# 计算平均局部能量矩阵
el_mat_sum = jnp.zeros((K, K), dtype=complex)
n_sample_final = samples_final.shape[0]

for xb in samples_final:
    _, el_mat = compute_local_energy(ha, total_ansatz, xb)
    el_mat_sum += el_mat

el_mat_avg = el_mat_sum / n_sample_final

# 对角化 → 得到 K 个本征态能量（已排序）
eigen_energies = jnp.linalg.eigvalsh(el_mat_avg).real

# 输出结果
for i, e in enumerate(eigen_energies):
    exc_ev = (e - eigen_energies[0]) * 27.2114  # 转换为 eV
    print(f"NES-VMC  E{i} = {e:.8f} Ha  |  激发能: {exc_ev:.4f} eV")

# FCI 对比
print("\n" + "-"*60)
print("FCI 精确结果对比：")
for i in range(K):
    exc_ev = (E_fcis[i] - E_fcis[0]) * 27.2114
    print(f"FCI      E{i} = {E_fcis[i]:.8f} Ha  |  激发能: {exc_ev:.4f} eV")

In [ ]:
model1 = total_ansatz.single_ansatz_list[0]
model2 = total_ansatz.single_ansatz_list[1]


In [ ]:
model1(jnp.array([1,0,1,0]))

In [ ]:
model2(jnp.array([1,0,1,0]))

In [ ]:
hi.all_states()